# AG News RAG Retrieval Pipeline

This notebook demonstrates an end-to-end retrieval pipeline for an AG News vector store.

The pipeline does five main things:

1. Resolves the vector store ID dynamically by matching the vector store `name` plus vector-store-level metadata: `tenant_id` and `version_no`.
2. Uses an LLM function call to extract AG News metadata filters from the user query.
3. Converts the extracted metadata into a LlamaStack vector store filter.
4. Runs hybrid search against the resolved vector store.
5. Calls the LlamaStack rerank endpoint to reorder candidates before generating the final RAG answer.

Sample query:

```text
Find business news about oil prices
```


## 1. Install dependencies

Run this cell if your notebook environment does not already have the required packages installed.

- `llama-stack-client` is used for chat completions and vector store APIs.
- `requests` is used to call the current LlamaStack rerank endpoint directly.


In [ ]:
# Uncomment if needed
%pip install requests llama-stack-client==0.6.1 


## 2. Imports and configuration

Update these values for your environment:

- `LLAMASTACK_URL`: your LlamaStack endpoint.
- `GEN_MODEL`: the generation model used for metadata extraction and final answer generation.
- `VECTOR_STORE_NAME`: the human-readable vector store name created by the ingestion notebook.
- `TENANT_ID`: vector-store-level metadata value used to select the correct store.
- `VERSION_NO`: vector-store-level metadata value used to select the correct store.
- `RERANKER_MODEL`: the reranker model registered in LlamaStack.

This notebook does **not** hardcode the vector store ID. It resolves it at runtime from the vector store catalog.


In [ ]:
import json
from typing import Any
import os
import requests
from llama_stack_client import LlamaStackClient

LLAMASTACK_URL = "http://lsd-service.agnews-rag-demo.svc.cluster.local:8321"
API_KEY = "EMPTY"

GEN_MODEL = "vllm-inference/llama-32-3b-instruct"

# Vector store lookup criteria.
# These values should match the ingestion notebook that created the vector store.
VECTOR_STORE_NAME = os.getenv(
    "VECTOR_STORE_NAME",
    "ag_news",
)
VECTOR_STORE_TENANT_ID = "DEV"
VECTOR_STORE_VERSION_NO = "v1"

# Reranker model registered in LlamaStack.
RERANKER_MODEL = "vllm-reranker/qwen3-reranker"

AG_NEWS_CATEGORIES = ["world", "sports", "business", "sci_tech"]

client = LlamaStackClient(
    base_url=LLAMASTACK_URL,
    api_key=API_KEY,
)

tools = [
    {
        "type": "function",
        "function": {
            "name": "build_ag_news_metadata_filter",
            "description": "Extract metadata filters from a natural language AG News search query.",
            "parameters": {
                "type": "object",
                "properties": {
                    "category": {
                        "type": "string",
                        "enum": AG_NEWS_CATEGORIES,
                        "description": "AG News category. Allowed values: world, sports, business, sci_tech.",
                    },
                },
                "required": [],
            },
        },
    }
]


## 3. Generic response helpers

Different LlamaStack client versions may return dictionaries, Pydantic models, or SDK objects. These helpers normalize objects into dictionaries and safely extract nested values.


In [ ]:
def to_dict(obj: Any) -> dict[str, Any]:
    """Convert SDK/Pydantic/dict responses to a plain dictionary."""
    if obj is None:
        return {}

    if isinstance(obj, dict):
        return obj

    if hasattr(obj, "model_dump"):
        return obj.model_dump()

    if hasattr(obj, "dict"):
        return obj.dict()

    if hasattr(obj, "__dict__"):
        return vars(obj)

    raise TypeError(f"Unsupported response type: {type(obj)}")


def get_nested_value(obj: Any, *candidate_keys: str) -> Any:
    """Return the first non-empty value found across several possible keys/attributes."""
    data = to_dict(obj) if not isinstance(obj, dict) else obj

    for key in candidate_keys:
        if key in data and data[key] not in (None, ""):
            return data[key]

    return None


def as_list_response_items(response: Any) -> list[dict[str, Any]]:
    """Normalize list-style responses that may be returned as {data: [...]}, {vector_stores: [...]}, or a raw list."""
    if isinstance(response, list):
        return [to_dict(item) for item in response]

    data = to_dict(response)

    for key in ("data", "vector_stores", "items"):
        value = data.get(key)
        if isinstance(value, list):
            return [to_dict(item) for item in value]

    return []


def extract_text_from_search_item(item: dict[str, Any]) -> str:
    """Extract text from a vector store search result item."""
    content_items = item.get("content", [])

    if not content_items:
        return ""

    first_content = content_items[0]

    if isinstance(first_content, dict):
        return first_content.get("text", "")

    if hasattr(first_content, "text"):
        return first_content.text

    return str(first_content)


## 4. Resolve vector store ID by name, `tenant_id`, and `version_no`

The ingestion notebook should create vector stores with metadata like:

```python
metadata={
    "tenant_id": "tenant-demo",
    "version_no": "v1",
}
```

This function lists vector stores, then selects the store where all three values match:

- vector store name
- `metadata.tenant_id`
- `metadata.version_no`

This makes the retrieval notebook portable across environments where vector store IDs change.


In [ ]:
def resolve_vector_store_id(
    vector_store_name: str,
    tenant_id: str,
    version_no: str,
) -> str:
    """Resolve a vector store ID from name + vector-store-level metadata."""
    response = client.vector_stores.list()
    vector_stores = as_list_response_items(response)

    matches = []

    for store in vector_stores:
        store_id = get_nested_value(store, "id", "vector_store_id")
        store_name = get_nested_value(store, "name")
        metadata = get_nested_value(store, "metadata") or {}

        if not isinstance(metadata, dict):
            metadata = to_dict(metadata)

        if (
            store_name == vector_store_name
            and str(metadata.get("tenant_id")) == str(tenant_id)
            and str(metadata.get("version_no")) == str(version_no)
        ):
            matches.append(
                {
                    "id": store_id,
                    "name": store_name,
                    "metadata": metadata,
                }
            )

    if not matches:
        available = [
            {
                "id": get_nested_value(store, "id", "vector_store_id"),
                "name": get_nested_value(store, "name"),
                "metadata": get_nested_value(store, "metadata") or {},
            }
            for store in vector_stores
        ]

        raise ValueError(
            "No vector store matched the requested name and metadata.\n"
            f"Requested name={vector_store_name!r}, "
            f"tenant_id={tenant_id!r}, "
            f"version_no={version_no!r}\n"
            f"Available vector stores:\n"
            f"{json.dumps(available, indent=2, ensure_ascii=False)}"
        )

    if len(matches) > 1:
        raise ValueError(
            "More than one vector store matched the requested name and metadata. "
            "Use a unique name, tenant_id, and version_no combination.\n"
            f"Matches:\n"
            f"{json.dumps(matches, indent=2, ensure_ascii=False)}"
        )

    return matches[0]["id"]

## 5. Tool schema for metadata extraction

The tool definition constrains the LLM to return only valid AG News categories:

- `world`
- `sports`
- `business`
- `sci_tech`

The metadata extraction step does not answer the user. It only converts a natural-language question into structured metadata filters for retrieval.


## 6. Extract metadata filters from the query

For example, the query:

```text
Find business news about oil prices
```

should extract:

```json
{
  "category": "business"
}
```

The function returns an empty dictionary when no filter is confidently identified.


In [ ]:
def extract_ag_news_filters_from_query(user_query: str) -> dict[str, Any]:
    messages = [
        {
            "role": "system",
            "content": """
You extract metadata filters from user questions for an AG News vector store.

Only extract filters that are explicitly or clearly implied.
Do not invent metadata values.

AG News categories:
- world: international news, politics, war, diplomacy
- sports: football, tennis, basketball, Olympics, matches
- business: economy, markets, companies, finance
- sci_tech: technology, science, software, internet, space

Metadata fields:
- category
""",
        },
        {
            "role": "user",
            "content": user_query,
        },
    ]

    response = client.chat.completions.create(
        model=GEN_MODEL,
        messages=messages,
        tools=tools,
        tool_choice={
            "type": "function",
            "function": {"name": "build_ag_news_metadata_filter"},
        },
    )

    message = response.choices[0].message
    tool_calls = getattr(message, "tool_calls", None)

    if not tool_calls:
        return {}

    try:
        return json.loads(tool_calls[0].function.arguments)
    except Exception:
        return {}


## 7. Build a LlamaStack vector store filter

The extracted query metadata is converted into the filter structure expected by the vector store search API.

For one category filter, the output looks like this:

```json
{
  "type": "eq",
  "key": "category",
  "value": "business"
}
```

`tenant_id` and `version_no` are used above to choose the correct vector store. If you also stored them as file/chunk attributes during ingestion, you may add them to search-time filters as well.


In [ ]:
def build_vector_store_filter(filters: dict[str, Any]) -> dict[str, Any] | None:
    conditions = []

    if filters.get("category"):
        conditions.append(
            {
                "type": "eq",
                "key": "category",
                "value": filters["category"],
            }
        )

    if not conditions:
        return None

    if len(conditions) == 1:
        return conditions[0]

    return {
        "type": "and",
        "filters": conditions,
    }


## 8. Rerank retrieved candidates with LlamaStack

Vector or hybrid search is fast and good for candidate retrieval, but it can still return chunks that are semantically close without being the best answer.

The reranking step sends the user query and candidate chunks to the reranker model. The reranker scores each query-document pair directly, then the notebook reorders the original vector search results by reranker score.

This code calls:

```text
POST /v1alpha/inference/rerank
```

The function keeps both scores:

- `original_vector_score`: score from vector/hybrid search
- `rerank_score`: score from the cross-encoder reranker


In [ ]:
def rerank_with_llamastack(
    query: str,
    search_results: dict[str, Any],
    max_num_results: int = 5,
) -> dict[str, Any]:
    """
    Call LlamaStack /v1alpha/inference/rerank after vector store search
    and return the original search items reordered by reranker score.
    """

    original_items = search_results.get("data", [])

    if not original_items:
        return {
            **search_results,
            "data": [],
            "rerank_response": None,
        }

    rerank_items = []

    for item in original_items:
        text = extract_text_from_search_item(item)

        # The endpoint schema supports either plain strings or typed text objects.
        # Typed text objects are clearer and safer.
        rerank_items.append(
            {
                "type": "text",
                "text": text,
            }
        )

    payload = {
        "model": RERANKER_MODEL,
        "query": query,
        "items": rerank_items,
        "max_num_results": max_num_results,
    }

    headers = {
        "Content-Type": "application/json",
    }

    if API_KEY:
        headers["Authorization"] = f"Bearer {API_KEY}"

    response = requests.post(
        f"{LLAMASTACK_URL.rstrip('/')}/v1alpha/inference/rerank",
        headers=headers,
        json=payload,
        timeout=60,
    )

    response.raise_for_status()
    rerank_response = response.json()

    reranked_items = []

    # Expected shape is usually:
    # {"data": [{"index": 0, "score": 0.92}]}
    # Some rerank APIs may return:
    # {"results": [{"index": 0, "relevance_score": 0.92}]}
    rerank_results = rerank_response.get("data") or rerank_response.get("results") or []

    for rerank_item in rerank_results:
        original_index = rerank_item.get("index")

        if original_index is None:
            continue

        if original_index < 0 or original_index >= len(original_items):
            continue

        original_item = original_items[original_index].copy()

        rerank_score = rerank_item.get("score") or rerank_item.get("relevance_score")

        original_item["rerank_score"] = rerank_score
        original_item["rerank_index"] = original_index
        original_item["original_vector_score"] = original_item.get("score")

        # Make final score the reranker score for formatting/display.
        if rerank_score is not None:
            original_item["score"] = rerank_score

        reranked_items.append(original_item)

    return {
        **search_results,
        "data": reranked_items,
        "rerank_response": rerank_response,
    }


## 9. End-to-end retrieval function

This function combines the complete retrieval flow:

1. Resolve the vector store ID from `VECTOR_STORE_NAME`, `TENANT_ID`, and `VERSION_NO`.
2. Extract metadata filters from the query.
3. Build the vector store search filter.
4. Search the AG News vector store using hybrid search.
5. Rerank the top candidate results.
6. Return a structured object that includes the resolved vector store ID, filters, retrieval settings, raw reranker response, and final reranked results.


In [ ]:
def search_ag_news_vector_store(
    user_query: str,
    candidate_results: int = 30,
    final_results: int = 5,
) -> dict[str, Any]:
    vector_store_id = resolve_vector_store_id(
        vector_store_name=VECTOR_STORE_NAME,
        tenant_id=VECTOR_STORE_TENANT_ID,
        version_no=VECTOR_STORE_VERSION_NO,
    )

    extracted_filters = extract_ag_news_filters_from_query(user_query)
    vector_filter = build_vector_store_filter(extracted_filters)

    search_kwargs = {
        "vector_store_id": vector_store_id,
        "query": user_query,
        "max_num_results": candidate_results,
        "search_mode": "hybrid",
        "rewrite_query": False,
    }

    if vector_filter:
        search_kwargs["filters"] = vector_filter

    response = client.vector_stores.search(**search_kwargs)
    search_results = to_dict(response)
    #print(json.dumps(search_results, indent=2, ensure_ascii=False))
    reranked_results = rerank_with_llamastack(
        query=user_query,
        search_results=search_results,
        max_num_results=final_results,
    )

    return {
        "user_query": user_query,
        "vector_store_lookup": {
            "name": VECTOR_STORE_NAME,
            "tenant_id": VECTOR_STORE_TENANT_ID,
            "version_no": VECTOR_STORE_VERSION_NO,
            "resolved_vector_store_id": vector_store_id,
        },
        "extracted_filters": extracted_filters,
        "vector_store_filter": vector_filter,
        "retrieval": {
            "search_mode": search_kwargs["search_mode"],
            "candidate_results": candidate_results,
            "reranker_model": RERANKER_MODEL,
            "final_results": final_results,
        },
        "results": reranked_results,
    }


## 10. Format retrieved context and generate the final RAG answer

The final answer is generated only from the retrieved and reranked search results.

The prompt asks the model to include rerank scores for transparency, which is useful when debugging retrieval quality or explaining why certain chunks were selected.


In [ ]:
def format_search_results_for_llm(results_json: dict[str, Any]) -> str:
    context_blocks = []

    for idx, item in enumerate(results_json.get("data", []), start=1):
        rerank_score = item.get("rerank_score")
        original_vector_score = item.get("original_vector_score")
        attributes = item.get("attributes", {})
        text = extract_text_from_search_item(item)

        context_blocks.append(
            f"""
Result {idx}
Rerank score: {rerank_score}
Original vector/hybrid score: {original_vector_score}
Metadata: {json.dumps(attributes, ensure_ascii=False)}
Content:
{text}
""".strip()
        )

    return "\n\n---\n\n".join(context_blocks)


def generate_rag_answer(user_query: str, search_output: dict[str, Any]) -> str:
    context = format_search_results_for_llm(search_output["results"])

    messages = [
        {
            "role": "system",
            "content": """
You are a RAG assistant for AG News.

Answer the user using only the retrieved and reranked search results.
Include the rerank score for each result you used.
If the results are not enough, say that the retrieved context is insufficient.

Return:
1. Short answer
2. Supporting results with rerank score
""",
        },
        {
            "role": "user",
            "content": f"""
User question:
{user_query}

Vector store lookup:
{json.dumps(search_output["vector_store_lookup"], indent=2, ensure_ascii=False)}

Extracted metadata filters:
{json.dumps(search_output["extracted_filters"], indent=2, ensure_ascii=False)}

Vector store filter:
{json.dumps(search_output["vector_store_filter"], indent=2, ensure_ascii=False)}

Retrieval configuration:
{json.dumps(search_output["retrieval"], indent=2, ensure_ascii=False)}

Retrieved and reranked results:
{context}
""",
        },
    ]

    response = client.chat.completions.create(
        model=GEN_MODEL,
        messages=messages,
        temperature=0.2,
    )

    return response.choices[0].message.content

## 11. Smoke test: list vector stores

This optional cell confirms that the client can connect to LlamaStack and list available vector stores.


In [ ]:
vector_stores_response = client.vector_stores.list()
vector_stores = as_list_response_items(vector_stores_response)

print("Available vector stores:")
print(json.dumps(vector_stores, indent=2, ensure_ascii=False))


## 12. Smoke test: resolve the vector store ID

This cell verifies that the notebook can find the correct vector store by `VECTOR_STORE_NAME`, `TENANT_ID`, and `VERSION_NO`.


In [ ]:
resolved_vector_store_id = resolve_vector_store_id(
    vector_store_name=VECTOR_STORE_NAME,
    tenant_id=VECTOR_STORE_TENANT_ID,
    version_no=VECTOR_STORE_VERSION_NO,
)

print("Resolved vector store ID:", resolved_vector_store_id)


## 13. Run retrieval for a sample query

The query below should trigger the `business` metadata filter and then retrieve AG News items related to oil prices.


In [ ]:
query = "Find business news about oil prices"

output = search_ag_news_vector_store(
    user_query=query,
    candidate_results=30,
    final_results=5,
)

print("Vector store lookup:")
print(json.dumps(output["vector_store_lookup"], indent=2, ensure_ascii=False))

print("\nExtracted filters:")
print(json.dumps(output["extracted_filters"], indent=2, ensure_ascii=False))

print("\nVector store filter:")
print(json.dumps(output["vector_store_filter"], indent=2, ensure_ascii=False))

print("\nRetrieval config:")
print(json.dumps(output["retrieval"], indent=2, ensure_ascii=False))


## 14. Inspect the raw reranker response

Use this cell to verify the exact rerank response shape returned by your LlamaStack version and reranker provider.


In [ ]:

print(json.dumps(output["results"].get("data"), indent=2, ensure_ascii=False))


## 15. Inspect reranked metadata and scores

This view is useful for comparing the original vector/hybrid score with the reranker score.


In [ ]:
for item in output["results"].get("data", []):
    print(
        {
            "rerank_score": item.get("rerank_score"),
            "original_vector_score": item.get("original_vector_score"),
            "attributes": item.get("attributes", {}),
        }
    )


## 16. Generate the final RAG answer

This final step sends the reranked context to the generation model and asks it to answer using only the retrieved results.


In [ ]:
rag_answer = generate_rag_answer(query, output)

print("Final RAG answer:")
print(rag_answer)
